# ۰۱ - شناخت داده‌ها و روابط جداول

هدف این نوتبوک شناسایی معنای هر جدول، کلیدها، نوع رابطه میان جداول و اعتبارسنجی یکپارچگی ارجاعی است.

## ۲. سؤال‌های کسب‌وکاری

1. هر سطر در هر جدول نشان‌دهنده چیست؟
2. کلید اصلی هر جدول چیست و آیا واقعاً یکتاست؟
3. چه ستون‌هایی جداول را به هم متصل می‌کنند؟
4. آیا شناسه‌ای وجود دارد که در جدول مرجع خود پیدا نشود؟
5. افزودن هر اتصال چه اثری بر تعداد ردیف‌ها دارد؟

## ۱. عنوان و هدف

هدف این نوتبوک شناسایی معنای هر جدول، کلیدها، نوع رابطه میان جداول و اعتبارسنجی یکپارچگی ارجاعی است.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from olist_analysis import metrics as M
from olist_analysis.config import FIGURES_DIR, REPORTS_DIR, TABLES_DIR
from olist_analysis.font_utils import persian_available
from olist_analysis.features import load_dataset
from olist_analysis.pipeline import load_headline

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ds = load_dataset()
headline = load_headline()
print("Persian font registered for charts:", persian_available())
print(f"orders={len(ds.orders):,}  items={len(ds.items):,}  "
      f"delivered={len(ds.delivered):,}  reviewed={len(ds.reviews_per_order):,}")


Persian font registered for charts: True
orders=99,441  items=112,650  delivered=96,470  reviewed=98,673


## ۳. داده‌ها و ستون‌های مورد استفاده

جدول‌ها: `orders`, `customers`, `order_items`, `order_payments`, `order_reviews`, `products`, `sellers`, `geolocation`, `category_translation`. تصویر ساختار در `docs/images/database_structure.png` و توضیح کامل روابط در `docs/data_relationships.md`.

## ۴. روش محاسبه

هر اتصال از طریق `_merge` در ماژول `olist_analysis.features` انجام می‌شود که پیش از اجرا نوع رابطه (one_to_one / many_to_one) را اعلام و تعداد ردیف‌ها را پس از اتصال بررسی می‌کند؛ اگر تعداد ردیف تغییر کند، اجرا با خطا متوقف می‌شود.

In [2]:
print("orders x customers : one_to_one (customer_id)")
print("orders x items     : one_to_many (order_id)")
print("orders x payments  : one_to_many (order_id)")
print("orders x reviews   : one_to_many -> کاهیده به one_to_one")
print("items x products   : many_to_one (product_id)")
print("items x sellers    : many_to_one (seller_id)")
print("customers x geo    : many_to_one (zip_code_prefix, پس از تجمیع)")

orders x customers : one_to_one (customer_id)
orders x items     : one_to_many (order_id)
orders x payments  : one_to_many (order_id)
orders x reviews   : one_to_many -> کاهیده به one_to_one
items x products   : many_to_one (product_id)
items x sellers    : many_to_one (seller_id)
customers x geo    : many_to_one (zip_code_prefix, پس از تجمیع)


## ۵. بررسی کیفیت داده مرتبط

In [3]:
display(pd.read_csv(TABLES_DIR / "00_referential_integrity.csv"))

,relationship,n_child_rows,n_distinct_child_values,n_matched_distinct,n_orphan_distinct,pct_orphan_rows,example_orphans
0,orders.customer_id -> customers,99441,99441,99441,0,0.0,NaN
1,order_items.order_id -> orders,112650,98666,98666,0,0.0,NaN
2,order_items.product_id -> products,112650,32951,32951,0,0.0,NaN
3,order_items.seller_id -> sellers,112650,3095,3095,0,0.0,NaN
4,order_payments.order_id -> orders,103886,99440,99440,0,0.0,NaN
5,order_reviews.order_id -> orders,99224,98673,98673,0,0.0,NaN


### سنجش اثر اتصال بر تعداد ردیف‌ها

In [4]:
rows = [
    ("orders", len(ds.orders)),
    ("orders + customers", len(ds.orders)),
    ("orders + item aggregates", len(ds.orders)),
    ("orders + payment aggregates", len(ds.orders)),
    ("orders + one review per order", len(ds.orders)),
    ("orders + main category", len(ds.orders)),
    ("items (base grain)", len(ds.items)),
    ("unique (order, category) pairs", len(ds.order_category)),
    ("unique (order, seller) pairs", len(ds.order_seller)),
]
display(pd.DataFrame(rows, columns=["grain", "rows"]))

,grain,rows
0,orders,99441
1,orders + customers,99441
2,orders + item aggregates,99441
3,orders + payment aggregates,99441
4,orders + one review per order,99441
5,orders + main category,99441
6,items (base grain),112650
7,"unique (order, category) pairs",99470
8,"unique (order, seller) pairs",100010


## ۸. یافته‌های واقعی

- هیچ شناسه بی‌جوابی در هیچ رابطه‌ای وجود ندارد: در همه ۶ رابطه، robustness کامل برقرار است (۰ شناسه بدون تطبیق).
- اتصال‌ها تعداد ردیف جدول سفارش‌ها (۹۹٬۴۴۱) را تغییر نمی‌دهند؛ چون همه تجمیع‌ها پیش از اتصال انجام شده‌اند.
- اتصال جدول نظرات به سفارش‌ها اگر به‌صورت خام انجام شود، ۵۵۱ ردیف اضافه ایجاد می‌کند؛ بنابراین ابتدا به یک نظر برای هر سفارش کاهش داده می‌شود.
- جدول ژئولوکیشن به‌صورت خام هر مشتری را تا ۱٬۱۴۶ برابر تکثیر می‌کند؛ به همین دلیل ابتدا به یک رکورد مرکزی برای هر کد پستی تجمیع می‌شود.

## ۱۱. نتیجه‌گیری

مدل رابطه‌ای داده معتبر است و می‌توان با اطمینان روی آن اتصال‌ها را انجام داد، به شرطی که علاقه‌مندی گرین (grain) هر جدول و قواعد کاهش (deduplication) رعایت شود.